# P.R.I.S.M. - Backend Evaluation Notebook

This notebook is designed for judges and evaluators to test the P.R.I.S.M. (Probabilistic Reasoning and Interpretability System for Models) backend API. The frontend is deployed on Vercel, so this notebook only sets up and exposes the backend.

### Instructions:
1. Ensure your Kaggle notebook has the **T4 x2** accelerator enabled in the Session Options.
2. Ensure **Internet** is toggled **On**.
3. Run all the cells below in order.
4. The final cell will generate a public URL for the backend.
5. **Copy the backend URL** and provide it to the Vercel frontend at: [Your Vercel Frontend URL]
6. *Note: If LocalTunnel prompts you for an "Endpoint IP", copy and paste the IP address printed right above the link.*

In [1]:
# 1. Setup Environment & Clone Repository
!echo "Installing Node.js..."
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

!echo "\nCloning P.R.I.S.M. repository..."
!git clone https://github.com/chandan989/P.R.I.S.M..git

Installing Node.js...
2026-05-18 14:43:37 - Installing pre-requisites
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     mm
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]      
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]m
Get:12 https://developer

In [2]:
# 2. Install llama-cpp-python and Download Model
!echo "Installing llama-cpp-python with CUDA 12.1 support..."
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 huggingface_hub

Installing llama-cpp-python with CUDA 12.1 support...
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 926.8 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00


In [ ]:
!echo "\nDownloading P.R.I.S.M. MXFP4 GGUF model..."
# !mkdir -p models

!pip install -U "huggingface_hub[cli]"

import os
from huggingface_hub import snapshot_download

print("\nDownloading P.R.I.S.M. MXFP4 GGUF model...")
os.makedirs("models", exist_ok=True)

snapshot_download(
    repo_id="chandan989/prism-gemma-4-26B-A4B-it-MXFP4-v3.5",
    local_dir="models",
)
# snapshot_download(repo_id=repo_id, local_dir="/content/model")

\nDownloading P.R.I.S.M. MXFP4 GGUF model...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 43.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 35.2 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1



Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# 3. Configure and Start the FastAPI Backend
import subprocess
import time
import os

print("Installing Python backend dependencies...")
!pip install -q -r P.R.I.S.M./backend/requirements.txt

print("\nConfiguring environment variables and starting FastAPI on port 8000...")

# 1. Inherit the base environment from the Kaggle kernel
server_env = os.environ.copy()

# 2. Inject your specific configurations directly into the dictionary
# This completely overrides the need for a physical .env file
server_env["MODEL_BACKEND"] = "llama_cpp"
server_env["MODEL_PATH"] = "../../models/"
server_env["KB_ROOT"] = "../knowledge_base"

# 3. Spawn the server with the modified environment and capture logs
log_file = open("backend_server.log", "w")
backend_process = subprocess.Popen(
    ["python3", "server.py", "--port", "8000"],
    cwd="P.R.I.S.M./backend",
    env=server_env,               # <--- Injects the variables dynamically
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait 30 seconds to allow the model to load into VRAM
time.sleep(30)
print("Startup sequence complete.")
print("Run '!tail -n 50 backend_server.log' in the next cell to verify status.")

In [ ]:
# 4. Expose Backend via Cloudflare Tunnel
import subprocess
import time

print("\n" + "="*60)
print("🚀 BACKEND READY: GENERATING CLOUDFLARE URL")
print("="*60)

# Download cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("Starting tunnel (this takes a few seconds)...\n")

# Bypass IPython's background process restriction using subprocess.Popen
log_file = open("cloudflared.log", "w")
tunnel_process = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for the tunnel to establish and generate the URL
time.sleep(8)

# Extract and display the URL
print("\033[1;32m=== YOUR BACKEND URL ===\033[0m")
!grep -o 'https://[-a-zA-Z0-9]*\.trycloudflare\.com' cloudflared.log | head -n 1
print("\033[1;32m========================\033[0m\n")

print("\033[1mNEXT STEPS:\033[0m")
print("1. Copy the URL above (it bypasses all password screens).")
print("2. Paste it into your Vercel frontend configuration.")
print("3. Begin testing P.R.I.S.M.")

In [ ]:
!tail -n 50 backend_server.log

# Frontend URL: https://p-r-i-s-m-pearl.vercel.app/